In [ ]:
%run initialize_environment.py

In [ ]:
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri("sqlite:///resources/Chinook.db")

In [16]:
@tool
def sql_query(query: str) -> str:

    """Obtain information from the database using SQL queries"""

    try:
        return db.run(query)
    except Exception as e:
        return f"Error: {e}"

sql_query.invoke("SELECT * FROM Artist LIMIT 10")

"[(1, 'AC/DC'), (2, 'Accept'), (3, 'Aerosmith'), (4, 'Alanis Morissette'), (5, 'Alice In Chains'), (6, 'Antônio Carlos Jobim'), (7, 'Apocalyptica'), (8, 'Audioslave'), (9, 'BackBeat'), (10, 'Billy Cobham')]"

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    model=create_azure_llm(),
    tools=[sql_query]
)

In [7]:
messages = [
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response = agent.invoke({"messages": messages})
pprint(response['messages'][-1].content)

("It seems there is no table named 'artists' in the database. Could you please "
 'provide the correct table name or more details about the structure of the '
 'database?')


In [8]:
from langchain.messages import HumanMessage, SystemMessage

schema_hint = """
Use only existing tables/columns from this SQLite schema:
Artist(ArtistId, Name)
Album(AlbumId, Title, ArtistId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.Quantity).
"""

messages = [
    SystemMessage(content=schema_hint),
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response = agent.invoke({"messages": messages})

In [ ]:
tool_call_msg = next(m for m in response['messages'] if getattr(m, 'tool_calls', None))
pprint(tool_call_msg.tool_calls[0])

{'args': {'query': 'SELECT Artist.Name, SUM(InvoiceLine.Quantity) AS '
                   'Popularity FROM Artist JOIN Album ON Artist.ArtistId = '
                   'Album.ArtistId JOIN Track ON Album.AlbumId = Track.AlbumId '
                   'JOIN InvoiceLine ON Track.TrackId = InvoiceLine.TrackId '
                   "WHERE Artist.Name LIKE 'S%' GROUP BY Artist.ArtistId ORDER "
                   'BY Popularity DESC LIMIT 1;'},
 'id': 'call_C5q7wyS5b8kuBUMEc4swUS42',
 'name': 'sql_query',
 'type': 'tool_call'}


In [9]:
response['messages'][-1].content

'The most popular artist beginning with \'S\' in this database is "Smashing Pumpkins," with a total of 24 tracks sold.'

## Dynamic schema in system message
This section demonstrates how to extract the schema at runtime and append it to the system message before invoking the agent.  But first, some python pre-requisite(s)...

In [11]:
# `ast` stands for Abstract Syntax Trees.  It is used to safely parse
# string-ified Python literals (e.g., arrays/list/tuple results) into actual Python objects.
# But it does not evaluate any python code in the string!

import ast

ast.literal_eval("[1, 2, 3]")
# → [1, 2, 3]

ast.literal_eval("{'a': 1, 'b': 2}")
# → {'a': 1, 'b': 2}

ast.literal_eval("(True, None, 3.14)")
# → (True, None, 3.14)


(True, None, 3.14)

In [12]:
def get_sqlite_schema_overview(database) -> str:
    """Build table(column1, column2, ...) lines from SQLite metadata."""
    table_rows = ast.literal_eval(database.run(
        """
            SELECT name
            FROM sqlite_master
            WHERE type = 'table' AND name NOT LIKE 'sqlite_%'
            ORDER BY name;
            """
        )
    )

    schema_lines = []
    for row in table_rows:
        table_name = row[0]
        
        # PRAGMA keyword tells SQLite to return metadata about the columns of the specified table.
        column_rows = ast.literal_eval(database.run(f"PRAGMA table_info('{table_name}')"))
        column_names = [col[1] for col in column_rows]
        if column_names:
            schema_lines.append(f"{table_name}({', '.join(column_names)})")

    return "\n".join(schema_lines)


schema_overview_dynamic = get_sqlite_schema_overview(db)
print(schema_overview_dynamic)

Album(AlbumId, Title, ArtistId)
Artist(ArtistId, Name)
Customer(CustomerId, FirstName, LastName, Company, Address, City, State, Country, PostalCode, Phone, Fax, Email, SupportRepId)
Employee(EmployeeId, LastName, FirstName, Title, ReportsTo, BirthDate, HireDate, Address, City, State, Country, PostalCode, Phone, Fax, Email)
Genre(GenreId, Name)
Invoice(InvoiceId, CustomerId, InvoiceDate, BillingAddress, BillingCity, BillingState, BillingCountry, BillingPostalCode, Total)
InvoiceLine(InvoiceLineId, InvoiceId, TrackId, UnitPrice, Quantity)
MediaType(MediaTypeId, Name)
Playlist(PlaylistId, Name)
PlaylistTrack(PlaylistId, TrackId)
Track(TrackId, Name, AlbumId, MediaTypeId, GenreId, Composer, Milliseconds, Bytes, UnitPrice)


In [13]:
dynamic_schema_hint = f"""
Use only existing tables/columns from this SQLite schema:
{schema_overview_dynamic}

Do not invent columns like artist_name or popularity.
If popularity is requested, define it as total tracks sold: SUM(InvoiceLine.Quantity).
"""

dynamic_messages = [
    SystemMessage(content=dynamic_schema_hint),
    HumanMessage(content="Who is the most popular artist beginning with 'S' in this database?")
]

response_dynamic = agent.invoke({"messages": dynamic_messages})

In [15]:
from pprint import pprint

dynamic_tool_call_msg = next(
    m for m in response_dynamic["messages"] if getattr(m, "tool_calls", None)
)
pprint(dynamic_tool_call_msg.tool_calls[0])

print("Generated SQL:")
pprint(dynamic_tool_call_msg.tool_calls[0]["args"]["query"])

print("\nFinal answer:")
print(response_dynamic["messages"][-1].content)

{'args': {'query': 'SELECT Artist.ArtistId, Artist.Name, '
                   'SUM(InvoiceLine.Quantity) AS Popularity FROM Artist JOIN '
                   'Album ON Artist.ArtistId = Album.ArtistId JOIN Track ON '
                   'Album.AlbumId = Track.AlbumId JOIN InvoiceLine ON '
                   'Track.TrackId = InvoiceLine.TrackId WHERE Artist.Name LIKE '
                   "'S%' GROUP BY Artist.ArtistId ORDER BY Popularity DESC "
                   'LIMIT 1;'},
 'id': 'call_2y7bFecqZPgyhtt09pcQGczQ',
 'name': 'sql_query',
 'type': 'tool_call'}
Generated SQL:
('SELECT Artist.ArtistId, Artist.Name, SUM(InvoiceLine.Quantity) AS Popularity '
 'FROM Artist JOIN Album ON Artist.ArtistId = Album.ArtistId JOIN Track ON '
 'Album.AlbumId = Track.AlbumId JOIN InvoiceLine ON Track.TrackId = '
 "InvoiceLine.TrackId WHERE Artist.Name LIKE 'S%' GROUP BY Artist.ArtistId "
 'ORDER BY Popularity DESC LIMIT 1;')

Final answer:
The most popular artist beginning with 'S' in this database is "S